In [20]:
from dotenv import load_dotenv # import the load_dotenv function from the dotenv library
#  Dotenv is a small tool and convention for managing configuration values (like API keys, database URLs, and secrets) in a simple text file instead of hard‑coding them in your code.

load_dotenv() # call the load_dotenv function to load environment variables from a .env file

True

In [21]:
from langchain.tools import tool # Allows you to create a function that can be called by the agent. This is how you can add external tools to your agent, like a web search tool or a calculator.
from typing import Dict, Any
from tavily import TavilyClient # Import the TavilyClient class from the tavily library. This is the client that will allow us to interact with the Tavily API.

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [31]:
import json, os

PROFILE_FILE = "user_profile.json"

def load_profile():
    if os.path.exists(PROFILE_FILE):
        with open(PROFILE_FILE) as f:
            return json.load(f)
    return {"allergies": [], "cuisine_preferences": []}

def save_profile(profile):
    with open(PROFILE_FILE, "w") as f:
        json.dump(profile, f, indent=2)

def build_system_prompt():
    profile = load_profile()
    allergies = ", ".join(profile["allergies"]) if profile["allergies"] else "none"
    cuisines = ", ".join(profile["cuisine_preferences"]) if profile["cuisine_preferences"] else "no preference"

    return f"""You are a personal chef assistant.

User profile:
- Allergies: {allergies}
- Preferred cuisines: {cuisines}

The user will give you ingredients they have. Search the web for recipes using only those ingredients.
If some ingredients conflict with the user's allergens, silently ignore those ingredients and work with the remaining safe ones.
Never ask the user to provide more ingredients — always work with what they gave you unless nothing is working.
"""

print("Profile loaded:", load_profile())


Profile loaded: {'allergies': [], 'cuisine_preferences': ['Italian', 'Asian']}


In [32]:
from langchain.tools import tool

@tool
def save_allergies(allergies: str) -> str:
    """Save the user's food allergies. Input should be comma-separated allergies e.g. 'nuts, dairy, shellfish'"""
    profile = load_profile()
    profile["allergies"] = [a.strip() for a in allergies.split(",")]
    save_profile(profile)
    return f"Saved allergies: {allergies}"

@tool
def save_cuisine_preferences(cuisines: str) -> str:
    """Save the user's preferred cuisine styles. Input should be comma-separated e.g. 'Italian, Japanese'"""
    profile = load_profile()
    profile["cuisine_preferences"] = [c.strip() for c in cuisines.split(",")]
    save_profile(profile)
    return f"Saved cuisine preferences: {cuisines}"


In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

agent = create_agent(
    model=model,
    tools=[web_search, save_allergies, save_cuisine_preferences],  # added profile tools
    system_prompt=build_system_prompt(),
    checkpointer=InMemorySaver()
)


In [49]:
import base64
from langchain.messages import HumanMessage
from ipywidgets import FileUpload
import ipywidgets as widgets
from IPython.display import display

config = {"configurable": {"thread_id": "1"}}

def chat(message, image_bytes=None, image_type="image/png"):
    if image_bytes:
        # multimodal message: text + image
        base64_image = base64.b64encode(image_bytes).decode("utf-8")
        content = [
            {"type": "text", "text": message},
            {"type": "image_url", "image_url": {"url": f"data:{image_type};base64,{base64_image}"}}
        ]
    else:
        content = message  # text only, same as before

    response = agent.invoke(
        {"messages": [HumanMessage(content=content)]},
        config
    )
    print(response["messages"][-1].content)


In [50]:
uploader = FileUpload(accept='.png,.jpeg,.jpg', multiple=False)
display(uploader)

def chat_with_image(message):
    if uploader.value:
        # ipywidgets 8.x returns a tuple, not a dict
        file = uploader.value[0]
        image_bytes = bytes(file["content"])
        image_type = file.get("type", "image/png")
        chat(message, image_bytes=image_bytes, image_type=image_type)
    else:
        chat(message)
  # no image, text only
# Example usage:
#chat_with_image("What ingredients do you see in this image? What can I make for dinner?")


FileUpload(value=(), accept='.png,.jpeg,.jpg', description='Upload')

In [47]:
# First run — save your profile
chat_with_image("I'm allergic to dairy and nuts, and I love Italian and Asian food")


You've already told me about your allergies and preferences. Is there anything else I can help you with?


In [ ]:
# Asking based on the image and profile
chat_with_image("What ingredients do you see in this image? What can I make for dinner?")


In the image, I can see Feta cheese and yogurt. There also appears to be some sort of stuffed vegetable dish on a plate.

Since you are allergic to dairy, I cannot recommend any recipes that use the feta cheese or yogurt.

Do you have any other ingredients you'd like to use, or would you like me to search for recipes that use stuffed vegetables?


In [ ]:
# Follow up with just text — it should remember your profile and not recommend anything with dairy or nuts, and prioritize Italian and Asian dishes
chat("Yes, please search for recipes that use stuffed vegetables")


I can't search for recipes based on "stuffed vegetables" alone, as it's a broad category. Could you please tell me what kind of vegetables are stuffed, or any other ingredients you have on hand?


In [ ]:
# Follow up again, reminding it it can use the image and profile info
chat("you can use the one you saw in the image or something i told you about in the past")

I found a recipe for "Stuffed Eggplant with Sweet Potatoes" that also includes tomatoes. It seems like a good option given the ingredients you have.

Would you like me to provide you with the recipe for "Stuffed Eggplant with Sweet Potatoes"?


In [54]:
chat("YES!")

Here is a recipe for Stuffed Eggplant with Sweet Potatoes and Tomatoes:

**Ingredients:**

*   2 medium eggplants
*   2 medium sweet potatoes, peeled and cubed
*   1-2 tomatoes, chopped
*   Olive oil
*   Salt and pepper to taste
*   Optional: herbs like parsley or cilantro

**Instructions:**

1.  Preheat your oven to 400°F (200°C).
2.  Cut the eggplants in half lengthwise and scoop out most of the flesh, leaving about a 1/2-inch border. Chop the scooped-out eggplant flesh.
3.  Boil or steam the cubed sweet potatoes until tender.
4.  In a pan, heat a little olive oil and sauté the chopped eggplant flesh and chopped tomatoes until softened.
5.  In a bowl, combine the cooked sweet potatoes, sautéed eggplant and tomato mixture. Season with salt and pepper.
6.  Stuff the eggplant halves with the sweet potato and vegetable mixture.
7.  Place the stuffed eggplants in a baking dish with a little water or broth at the bottom.
8.  Bake for 25-30 minutes, or until the eggplant is tender and the f